# 🎯 [QLT04_M08] 전군 장비 가동률 데이터 일괄 분석 스트레스 테스트 및 성능 검증
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M08 |
| **미션 주제** | 전군 장비 가동률 데이터 일괄 분석 스트레스 테스트 및 성능 검증 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 전군 장비 가동률 데이터 일괄 분석 스트레스 테스트 및 성능 검증
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전군 장비 가동률 데이터 일괄 분석 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **12개월치 가상 장비 가동률 자료를 월 단위 배치로 일괄 분석하고, 오류·장애 주입과 자료량 500% 부하에서 성능과 안정성을 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **월 배치를 적재-검증-분석-보고 4단계로 자동 실행하는 스케줄러 모사 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/pipeline/qlt04_batch_runner.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 수작업 대비 업무 소요 시간 80% 이상 절감 효과

#### 미션 PASS 조건

- 정량 기준: 전체 배치 파이프라인 실행 시간 5분 이내, 자동화 성공률 99% 이상


- 진행 기준: 스케줄러 연동 테스트 및 자동 실행 확인


- 검증 방법: 실제 월간 업무 데이터 1년치 일괄 배치 시뮬레이션


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""12개월치 가상 장비 가동률 자료를 월 단위 배치(적재 -> 검증·격리 -> 룰·모델 분석 -> 보고)로 자동 실행하는 스케줄러 모사를 만들고, 오류 레코드·일시 장애 주입과 자료량 100%·200%·500% 부하에서 실행 시간·성공률·자료 유실률·수작업 대비 절감을 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 정비·검사 병목 판별 룰 설정(YAML 형식 문자열: 기준이 바뀌면 코드가 아니라 이 설정만 고친다) ----
# clause는 룰과 대응시킨 '가상 품질지침 조항' 예시이며 실제 규정 원문이 아니다.
RULES_YAML = """
- {id: Q01, name: 부품 재고 가용률 50% 미만, field: parts_stock_pct, op: "<", value: 50, weight: 2, clause: 가상품질지침 제3조}
- {id: Q02, name: 정비병 충원율 60% 미만, field: mechanic_fill_pct, op: "<", value: 60, weight: 2, clause: 가상품질지침 제4조}
- {id: Q03, name: 부품 대기 8일 초과, field: parts_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제5조}
- {id: Q04, name: 정비병 부족 대기 8일 초과, field: labor_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제6조}
- {id: Q05, name: 외주 정비 15일 초과, field: outsource_days, op: ">", value: 15, weight: 2, clause: 가상품질지침 제7조}
- {id: Q06, name: 외주 검사 대기 6일 초과, field: inspect_wait_days, op: ">", value: 6, weight: 3, clause: 가상품질지침 제8조}
- {id: Q07, name: 재작업 1건 이상, field: rework_count, op: ">=", value: 1, weight: 3, clause: 가상품질지침 제9조}
- {id: Q08, name: 재작업률 25% 초과, field: rework_rate, op: ">", value: 0.25, weight: 2, clause: 가상품질지침 제9조}
- {id: Q09, name: 규격·제원 불일치 3건 이상, field: spec_mismatch, op: ">=", value: 3, weight: 3, clause: 가상품질지침 제10조}
- {id: Q10, name: 하자·불만 3건 이상, field: defect_reports, op: ">=", value: 3, weight: 2, clause: 가상품질지침 제11조}
- {id: Q11, name: 장비 연령 15년 초과, field: age_years, op: ">", value: 15, weight: 1, clause: 가상품질지침 제12조}
- {id: Q12, name: 정비 요청 6건 이상, field: maint_orders, op: ">=", value: 6, weight: 1, clause: 가상품질지침 제13조}
- {id: Q13, name: 대기 일수 합계 30일 초과, field: total_wait, op: ">", value: 30, weight: 3, clause: 가상품질지침 제14조}
- {id: Q14, name: 검사 대기가 외주 정비 일수의 80% 초과, field: insp_ratio, op: ">", value: 0.8, weight: 2, clause: 가상품질지침 제8조}
- {id: Q15, name: 규격 불일치 2건 이상과 하자 접수 동반, field: spec_defect, op: "==", value: true, weight: 3, clause: 가상품질지침 제10조}
- {id: Q16, name: 최근 90일 가동 시간 150시간 미만, field: op_hours, op: "<", value: 150, weight: 1, clause: 가상품질지침 제15조}
"""
GRADE_CUT = [(10, "고위험"), (6, "주의"), (3, "관심")]
OPS = {"<": lambda a, b: a < b, ">": lambda a, b: a > b, "<=": lambda a, b: a <= b, ">=": lambda a, b: a >= b, "==": lambda a, b: a == b}


def parse_rules(text):
    """YAML 한 줄 맵({key: value, ...}) 형식 룰 문자열을 읽는다(외부 yaml 라이브러리 없이)."""
    out = []
    for line in text.strip().splitlines():
        body = line.strip()[3:-1]
        r = {}
        for part in body.split(", "):
            k, v = part.split(": ", 1)
            v = v.strip().strip('"')
            r[k] = True if v == "true" else False if v == "false" else (float(v) if v.replace(".", "", 1).isdigit() else v)
        out.append(r)
    return out


def derive_rec(rec):
    """룰이 쓰는 파생 필드(대기 합계, 재작업률, 검사 대기 비율(외주 5일 이상일 때), 규격 불일치 2건 이상+하자 동반)를 추가한다."""
    r = dict(rec)
    r["total_wait"] = r["parts_wait_days"] + r["labor_wait_days"] + r["outsource_days"] + r["inspect_wait_days"]
    r["rework_rate"] = r["rework_count"] / r["maint_orders"] if r["maint_orders"] else 0.0
    r["insp_ratio"] = r["inspect_wait_days"] / r["outsource_days"] if r["outsource_days"] >= 5 else 0.0
    r["spec_defect"] = bool(r["spec_mismatch"] >= 2 and r["defect_reports"] >= 1)
    return r


def apply_rules(rec, rules, derived=False):
    """룰 적용: (점수, 적중 룰 id 목록). derived=True면 이미 파생 필드가 있는 레코드(경계 시험용)로 본다."""
    r = rec if derived else derive_rec(rec)
    hits = [x["id"] for x in rules if OPS[x["op"]](r[x["field"]], x["value"])]
    w = {x["id"]: x["weight"] for x in rules}
    return sum(w[h] for h in hits), hits


def grade_of(score):
    for cut, g in GRADE_CUT:
        if score >= cut:
            return g
    return "정상"

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """가동률(%) 회귀: 표준화 + 릿지 회귀."""
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(feat_frame(df), df["avail_pct"])


def fit_cause(df, cut=85, C=5):
    """가동률이 cut% 미만인 장비의 주요 저가동 요인(4종)을 분류한다."""
    low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cause_frame(low), low["main_cause"])


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

rules = parse_rules(RULES_YAML)
model = fit_avail(make_fleet(1200, seed=21))
BASE_N = 1000                                                   # 월 기준 장비 수(가상)
MANUAL_MIN = {"적재·정리": 0.5, "검증": 0.5, "분석": 1.5, "보고": 0.5}      # 장비 1대당 수작업 시간(분, 가정치)
CHECK_MIN = 6                                                   # 자동 분석 후 정비관이 고위험 1대를 확인하는 시간(분, 가정치)
rng = np.random.default_rng(8)
LOGS = []


def load(month, n, bad_rate=0.01):
    df = make_fleet(n, seed=100 + month)
    k = max(1, int(n * bad_rate))
    bad = rng.choice(n, k, replace=False)
    df["avail_pct"] = df["avail_pct"].astype(float)
    df.loc[bad[: k // 2], "avail_pct"] = np.nan                  # 결측
    df.loc[bad[k // 2:], "avail_pct"] = 150.0                    # 범위 오류
    return df


def validate(df):
    ok = df["avail_pct"].between(0, 100)
    return df[ok], df[~ok]


def analyze(df):
    recs = df.drop(columns=["main_cause"]).to_dict("records")
    sc = [apply_rules(r, rules)[0] for r in recs]
    pred = np.clip(model.predict(feat_frame(df)), 0, 100)
    return {"n": len(df), "avail": df["avail_pct"].mean(), "high": sum(s >= 10 for s in sc), "caution": sum(6 <= s < 10 for s in sc), "mae": np.mean(np.abs(pred - df["avail_pct"]))}


FAIL_ONCE = {(3, "분석"), (9, "적재")}                            # 일시 장애 주입(재시도 후 성공해야 함)
FAIL_ALWAYS = set()                                               # 지속 장애는 아래 별도 시험에서 주입(격리·알림 후 다음 달 계속)
tried = set()


def step(month, name, fn):
    for attempt in (1, 2):
        try:
            if (month, name) in FAIL_ONCE and (month, name) not in tried:
                tried.add((month, name)); raise IOError("temporary failure")
            if (month, name) in FAIL_ALWAYS:
                raise RuntimeError("persistent failure")
            return True, fn()
        except Exception as e:
            LOGS.append(f"{month:02d}월 {name} 시도 {attempt} 실패: {e}")
    return False, None


def run_batch(n, label, months=range(1, 13), verbose=False):
    t0 = time.perf_counter(); res, steps, ok_steps, quarantined, seen = [], 0, 0, 0, 0
    for m in months:
        ctx = {}
        for name, fn in [("적재", lambda: ctx.update(df=load(m, n))), ("검증", lambda: ctx.update(zip(("clean", "bad"), validate(ctx["df"])))),
                         ("분석", lambda: ctx.update(r=analyze(ctx["clean"]))), ("보고", lambda: ctx["r"])]:
            steps += 1
            ok, _ = step(m, name, fn)
            ok_steps += ok
            if not ok:
                break
        if "r" in ctx and ok:
            res.append({"월": m, **ctx["r"], "격리": len(ctx["bad"])})
        if "df" in ctx:
            seen += len(ctx["df"]); quarantined += len(ctx.get("bad", [])) if "bad" in ctx else 0
        if verbose:
            print(f"  진행 [{'#' * m}{'.' * (12 - m)}] {m:2d}/12월 {'완료' if ok else '실패(알림 발송, 다음 달 계속)'}")
    return pd.DataFrame(res), time.perf_counter() - t0, ok_steps, steps


# 1) 12개월 배치(스케줄러 모사)
print("[월 배치 실행: 장애 주입 포함]")
rs, dt, ok_s, steps = run_batch(BASE_N, "x1", verbose=True)
print(f"소요 {dt:.1f}초(목표 5분=300초 이내: {'달성' if dt < 300 else '미달'}), 단계 {ok_s}/{steps} 성공(일시 장애는 재시도로 복구) -> 자동화 성공률 {ok_s / steps:.1%} (목표 99% 이상: {'달성' if ok_s / steps >= 0.99 else '미달'})")
print("재시도·실패 로그:", " / ".join(LOGS[:3]), "...", f"(총 {len(LOGS)}건)")
print("[알림] 배치 완료: 월 12건 중 %d건 보고 생성, 실패 월 %s" % (len(rs), sorted(set(range(1, 13)) - set(rs["월"])) or "없음"))
print(rs.round(1).head(4).to_string(index=False), "\n  ...")

# 2) 지속 장애 시험(11월 보고 단계가 계속 실패): 해당 월만 건너뛰고 알림, 나머지 월은 계속(기능 축소 작동)
FAIL_ONCE.clear(); FAIL_ALWAYS.add((11, "보고")); LOGS.clear()
rp, dp, okp, stp = run_batch(BASE_N, "fault")
print(f"\n[지속 장애 시험] 단계 {okp}/{stp} 성공({okp / stp:.1%}), 보고 생성 {len(rp)}/12월, 실패 월 {sorted(set(range(1, 13)) - set(rp['월']))} -> 알림 발송, 다른 달은 정상 처리(배치 전체 중단 없음)")

# 3) 장애 없는 배치의 자료 유실 점검
FAIL_ALWAYS.clear(); LOGS.clear()
rs0, dt0, ok0, st0 = run_batch(BASE_N, "clean")
tot_in = BASE_N * 12
tot_out = int(rs0["n"].sum() + rs0["격리"].sum())
print(f"\n[장애 없는 배치] 단계 {ok0}/{st0} 성공({ok0 / st0:.1%}), 입력 {tot_in}건 = 분석 {int(rs0['n'].sum())} + 격리 {int(rs0['격리'].sum())} -> 유실 {tot_in - tot_out}건({(tot_in - tot_out) / tot_in:.1%})")

# 4) 부하 시험: 자료량 100%·200%·500%
print("\n[부하 시험: 월 자료량 증가 (12개월 전체)]")
lt = []
for mult in (1, 2, 5):
    r, d, o, s = run_batch(BASE_N * mult, f"x{mult}")
    lost = BASE_N * mult * 12 - int(r["n"].sum() + r["격리"].sum())
    lt.append((mult, d, o / s, lost))
    print(f"  자료 {mult * 100:3d}%: {BASE_N * mult * 12:6d}건, {d:5.1f}초, 성공 {o}/{s}, 유실 {lost}건, 초당 {BASE_N * mult * 12 / d:,.0f}건")
print(f"시스템 다운 0건, 데이터 유실 {sum(x[3] for x in lt)}건 (목표 0%: {'달성' if sum(x[3] for x in lt) == 0 else '미달'})")

# 5) 수작업 대비 절감(가정치 기반 상대 비교)
n_equip = BASE_N * 12
manual_h = n_equip * sum(MANUAL_MIN.values()) / 60
check_h = int(rs0["high"].sum()) * CHECK_MIN / 60
auto_h = dt0 / 3600 + check_h
print(f"\n[업무 시간 절감] 수작업 {manual_h:.0f}시간(장비당 {sum(MANUAL_MIN.values()):.1f}분) vs 자동 {auto_h:.1f}시간(실행 {dt0:.1f}초 + 고위험 {int(rs0['high'].sum())}대 확인 {check_h:.1f}시간) -> 절감 {1 - auto_h / manual_h:.0%} (목표 80% 이상: {'달성' if 1 - auto_h / manual_h >= 0.8 else '미달'})")
print("※ 시간 가정(장비당 수작업 분, 고위험 확인 6분)은 교육용 가정치이며 실제 업무 시간으로 바꿔 계산한다. 고위험 장비의 확인과 정비 우선순위 결정은 사람이 한다.")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(rs0["월"], rs0["avail"], marker="o"); ax[0].set_title("Monthly mean availability (synthetic)"); ax[0].set_xlabel("month")
ax[1].bar([f"x{x[0]}" for x in lt], [x[1] for x in lt], color="tab:blue"); ax[1].set_title("Batch time by data volume (sec)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[월 배치 실행: 장애 주입 포함]
  진행 [#...........]  1/12월 완료
  진행 [##..........]  2/12월 완료
  진행 [###.........]  3/12월 완료
  진행 [####........]  4/12월 완료
  진행 [#####.......]  5/12월 완료
  진행 [######......]  6/12월 완료
  진행 [#######.....]  7/12월 완료
  진행 [########....]  8/12월 완료
  진행 [#########...]  9/12월 완료
  진행 [##########..] 10/12월 완료
  진행 [###########.] 11/12월 완료
  진행 [############] 12/12월 완료
소요 0.3초(목표 5분=300초 이내: 달성), 단계 48/48 성공(일시 장애는 재시도로 복구) -> 자동화 성공률 100.0% (목표 99% 이상: 달성)
재시도·실패 로그: 03월 분석 시도 1 실패: temporary failure / 09월 적재 시도 1 실패: temporary failure ... (총 2건)
[알림] 배치 완료: 월 12건 중 12건 보고 생성, 실패 월 없음
 월   n  avail  high  caution  mae  격리
 1 990   79.1    80      103  4.0  10
 2 990   79.9    85       79  4.0  10
 3 990   78.6    86      108  4.0  10
 4 990   79.7    79       84  4.1  10 
  ...

[지속 장애 시험] 단계 47/48 성공(97.9%), 보고 생성 11/12월, 실패 월 [11] -> 알림 발송, 다른 달은 정상 처리(배치 전체 중단 없음)

[장애 없는 배치] 단계 48/48 성공(100.0%), 입력 12000건 = 분석 11880 + 격리 120 -> 유실 0건(0.0%)

[부하 시험: 월 자료량 증가 (12개월 전체)]
  자료 100%:  12000건,   0.3초, 성공 48/48, 유실 0건, 초당 37,904건
  자료 200%:  24000건,   0.6초, 성공 48/48, 유실 0건, 초당 41,507건
  자료 500%:  60000건,   1.1초, 성공 48/48, 유실 0건, 초당 52,896건
시스템 다운 0건, 데이터 유실 0건 (목표 0%: 달성)

[업무 시간 절감] 수작업 600시간(장비당 3.0분) vs 자동 98.3시간(실행 0.3초 + 고위험 983대 확인 98.3시간) -> 절감 84% (목표 80% 이상: 달성)
※ 시간 가정(장비당 수작업 분, 고위험 확인 6분)은 교육용 가정치이며 실제 업무 시간으로 바꿔 계산한다. 고위험 장비의 확인과 정비 우선순위 결정은 사람이 한다.
```
</details>
